In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
ddir = r"Z:\Home\rsenne\dCA1_Clean_Data\Anymaze"
animals = [ani for ani in os.listdir(ddir) if "astro" in ani]

animals.remove("astroF8") # removed as astrocytes are reactive
valid_session_bounds = (0, 330) # in seconds

time_vector = pd.read_csv(r"C:\Users\ryansenne\Desktop\Dill\Time.csv").iloc[:, 0].to_numpy()

In [4]:
def _parse_time_seconds(col):
    """
    Robustly parse a time column that may be either numeric seconds or
    'hh:mm:ss[.sss]' / 'mm:ss[.sss]' strings. Returns a float Series (seconds).
    """
    s = pd.Series(col)

    # If it already looks numeric, prefer that
    as_num = pd.to_numeric(s, errors="coerce")
    frac_num = as_num.notna().mean()

    # Heuristic: if >= 70% rows are numeric, treat as seconds
    if frac_num >= 0.7:
        return as_num.astype(float)

    # Otherwise, parse as clock-formatted strings (allow decimal comma)
    s_str = s.astype(str).str.strip().str.replace(",", ".", regex=False)
    td = pd.to_timedelta(s_str, errors="coerce")
    # If too many NaT, try to prepend "00:" to strings that look like "mm:ss"
    if td.isna().mean() > 0.3:
        needs_hour = s_str.str.count(":") == 1
        s_fix = s_str.mask(~needs_hour, s_str).where(~needs_hour, "00:" + s_str)
        td2 = pd.to_timedelta(s_fix.fillna(s_str), errors="coerce")
        td = td.fillna(td2)

    return td.dt.total_seconds()

def freezing_vector_from_csv(
    timestamps,
    csv_path,
    time_col="Time",
    freeze_col="Freezing",
    not_freeze_col="Not freezing",
    return_series=False,
    return_bouts=False
):
    """
    Build a per-timestamp freezing vector (1=freezing, 0=not) from a CSV of state changes.
    Handles time in seconds or 'hh:mm:ss[.sss]' / 'mm:ss[.sss]' strings.
    """

    df = pd.read_csv(csv_path)

    # Flexible column matching (strip/case/space-insensitive)
    def norm(name): return " ".join(str(name).strip().split()).lower()
    by_norm = {norm(c): c for c in df.columns}

    def pick(*cands):
        for c in cands:
            if norm(c) in by_norm:
                return by_norm[norm(c)]
        return None

    time_name = pick(time_col, "Time (s)", "time", "time (s)", "timestamp")
    if time_name is None:
        raise KeyError(f"Could not find a time column. Have columns: {list(df.columns)}")

    freeze_name = pick(freeze_col, "freezing")
    not_freeze_name = pick(not_freeze_col, "not freezing", "not_freezing")

    if freeze_name is None and not_freeze_name is None:
        raise KeyError(f"Need either '{freeze_col}' or '{not_freeze_col}' in the CSV.")

    # Parse time to seconds (float)
    t_seconds = _parse_time_seconds(df[time_name])

    # Build state
    if freeze_name is not None:
        state = pd.to_numeric(df[freeze_name], errors="coerce")
    else:
        state = 1 - pd.to_numeric(df[not_freeze_name], errors="coerce")

    # Clean and sort change points
    state_df = (
        pd.DataFrame({"t": t_seconds, "state": state})
        .dropna(subset=["t", "state"])
        .astype({"t": float, "state": int})
        .sort_values("t")
        .drop_duplicates(subset=["t"], keep="last")
        .reset_index(drop=True)
    )
    if state_df.empty:
        raise ValueError("No valid (time, state) rows after parsing/cleaning your CSV.")

    # Prepare timestamps (keep original order)
    ts = pd.Series(pd.to_numeric(pd.Series(timestamps), errors="coerce"), name="t")
    if ts.isna().all():
        raise ValueError("All provided `timestamps` are NaN after numeric coercion.")
    valid_mask = ~ts.isna()
    ts_valid = ts[valid_mask]

    # Sort for merge_asof
    order = np.argsort(ts_valid.values)
    ts_sorted = ts_valid.iloc[order].to_frame()

    aligned = pd.merge_asof(
        ts_sorted,
        state_df,   # already sorted by 't'
        on="t",
        direction="backward",
        allow_exact_matches=True,
    )

    # Fill initial portion before first change with first state
    first_state = int(state_df["state"].iloc[0])
    aligned["state"] = aligned["state"].fillna(first_state).astype(int)

    # Restore original order and shape
    out = np.empty(len(ts), dtype=int)
    out[:] = first_state
    out_valid = np.empty(len(ts_valid), dtype=int)
    out_valid[order.argsort()] = aligned["state"].to_numpy()
    out[valid_mask.to_numpy()] = out_valid

    bout_info = None
    if return_bouts:
        # Find transitions in state_df (0->1 is bout start, 1->0 is bout end)
        states = state_df['state'].values
        times = state_df['t'].values
        
        bouts = []
        if states[0] == 1:  # Started in freeze
            bout_start = times[0]
        else:
            bout_start = None
            
        for i in range(1, len(states)):
            if states[i] == 1 and states[i-1] == 0:  # Bout starts
                bout_start = times[i]
            elif states[i] == 0 and states[i-1] == 1:  # Bout ends
                if bout_start is not None:
                    bouts.append(times[i] - bout_start)
                    bout_start = None
        
        # Handle ongoing bout at end
        if states[-1] == 1 and bout_start is not None:
            bouts.append(times[-1] - bout_start)
        
        bout_info = {
            'n_bouts': len(bouts),
            'bout_durations': bouts,
            'total_freeze_time': sum(bouts) if bouts else 0.0
        }
    
    result = pd.Series(out, index=pd.Series(timestamps), name="freezing") if return_series else out
    
    return (result, bout_info) if return_bouts else result




## Freezing Vectors for Context A

In [5]:
freeze_vecs_cxt_a = {}
bout_info_cxt_a = {}
for ani in animals:
    animal_dir = os.path.join(ddir, ani)
    cxt_a = ani + "_cxta_" + "behavior.csv"
    try:
        df = pd.read_csv(os.path.join(animal_dir, cxt_a))
        freeze_vec, bout_info = freezing_vector_from_csv(
            time_vector,
            os.path.join(animal_dir, cxt_a),
            return_series=False,
            return_bouts=True,  
        )
        freeze_vecs_cxt_a[ani] = freeze_vec
        bout_info_cxt_a[ani] = bout_info
    except FileNotFoundError:
        continue

## Freezing Vectors For Context B

In [6]:
# get freeze vectors for each animal
freeze_vecs_cxt_b = {}
bout_info_cxt_b = {} 
for ani in animals:
    animal_dir = os.path.join(ddir, ani)
    cxt_b = ani + "_cxtb_" + "behavior.csv"
    try:
        df = pd.read_csv(os.path.join(animal_dir, cxt_b))
        freeze_vec, bout_info = freezing_vector_from_csv(
            time_vector,
            os.path.join(animal_dir, cxt_b),
            return_series=False,
            return_bouts=True,
        )
        freeze_vecs_cxt_b[ani] = freeze_vec
        bout_info_cxt_b[ani] = bout_info
    except FileNotFoundError:
        continue

In [47]:
# Extract metrics
freeze_percents_cxt_a = {ani: np.mean(vec) for ani, vec in freeze_vecs_cxt_a.items()}
freeze_percents_cxt_b = {ani: np.mean(vec) for ani, vec in freeze_vecs_cxt_b.items()}
n_bouts_cxt_a = {ani: info['n_bouts'] for ani, info in bout_info_cxt_a.items()}
n_bouts_cxt_b = {ani: info['n_bouts'] for ani, info in bout_info_cxt_b.items()} 

In [49]:
df = pd.DataFrame([freeze_percents_cxt_a, freeze_percents_cxt_b, n_bouts_cxt_a, n_bouts_cxt_b]).T
df.columns = ["Freezing_Pct_A", "Freezing_Pct_B", "N_Bouts_A", "N_Bouts_B"]
df.to_csv(r"C:\Users\ryansenne\Desktop\Dill\Freezing_Metrics.csv")

In [58]:
for ani, vec in freeze_vecs_cxt_a.items():
    np.savetxt(r"C:\Users\ryansenne\Desktop\Dill\cxta_traces_csv\{}_freeze_vec_cxt_a.csv".format(ani), vec, delimiter=",")

In [59]:
for ani, vec in freeze_vecs_cxt_b.items():
    np.savetxt(r"C:\Users\ryansenne\Desktop\Dill\cxtb_traces_csv\{}_freeze_vec_cxt_b.csv".format(ani), vec, delimiter=",")